# Hands-on Exercise 1 — Instrument & Compare Runs
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 1 (Experiment Tracking Deep Dive)

**Objective:** take an un-instrumented training script, add manual MLflow tracking, compare it against
autologging, and use `mlflow.search_runs()` to find the best run programmatically.

**Steps (from the slide deck):**
1. Take the provided starter script (`load_iris` → train a classifier — no MLflow calls yet).
2. Add `mlflow.set_experiment()` and wrap training in `with mlflow.start_run():`.
3. Log at least 3 hyperparameters and 2 metrics manually.
4. Re-run the script 4 times varying one hyperparameter (e.g. `n_estimators`: 10, 50, 100, 300).
5. Add `mlflow.autolog()` to a 5th run and compare what got captured automatically vs. manually.
6. Open the MLflow UI, select all 5 runs, and use **Compare** to find the best one.

**Deliverable:** a screenshot of the 5-run comparison view, plus the `run_id` of the best run found
via `mlflow.search_runs()`.

> **Prerequisite:** a local MLflow Tracking Server must already be running:
> ```bash
> mlflow server --backend-store-uri sqlite:///mlflow.db \
>     --default-artifact-root ./mlruns --host 0.0.0.0 --port 5000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:5000, http://127.0.0.1:5000"
> ```
> Run that command in a separate terminal *before* executing the cells below, then leave it running.

## Step 0 — Setup
Install dependencies (skip if already installed) and import libraries.

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp-comparison")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5000


## Step 1 — The starter script (un-instrumented)
This is the "before" version — plain scikit-learn, no tracking at all. Run it once just to confirm it works.

In [2]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_train, X_test, y_train, y_test = train_test_split(
    X / 255.0, y, test_size=0.2, random_state=42
)

def train_and_evaluate(hidden_layer_sizes=(100,), learning_rate_init=0.001, batch_size=128, max_iter=20):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        batch_size=batch_size,
        max_iter=max_iter,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return model, acc, f1

# Sanity check — no MLflow involved yet
_, acc, f1 = train_and_evaluate()
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=0.9755  f1_macro=0.9753


/home/srikar-j-v/courses/DA3408/assignment 1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


## Step 2 & 3 — Instrument it: manual logging
Wrap training in `with mlflow.start_run():` and log parameters, metrics, and a tag.

In [ ]:
def train_and_log(hidden_layer_sizes=(100,), learning_rate_init=0.001, batch_size=128, max_iter=20, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters (at least 3) ---
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("max_iter", max_iter)

        model, acc, f1 = train_and_evaluate(hidden_layer_sizes, learning_rate_init, batch_size, max_iter)

        # --- metrics (at least 2) ---
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model")

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  acc={acc:.4f}  f1={f1:.4f}")
        return run_id

baseline_run_id = train_and_log(
    hidden_layer_sizes=(100,),
    learning_rate_init=0.001,
    batch_size=256,
    max_iter=15,
    run_name="mlp-baseline"
)

/home/srikar-j-v/courses/DA3408/assignment 1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Logged run 5aefc6657f7541f897feda6cc85d6486  |  acc=0.0909  f1=0.0167
🏃 View run rf-baseline at: http://localhost:5000/#/experiments/1/runs/5aefc6657f7541f897feda6cc85d6486
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Step 4 — Sweep: 4 runs varying `n_estimators`
Re-run with different values and log each as its own MLflow run.

In [ ]:
sweep_run_ids = []
architectures = [(50,), (100,), (100, 50)]
learning_rates = [0.001, 0.01]

for layers in architectures:
    for lr in learning_rates:
        layer_str = "x".join(map(str, layers))
        run_name = f"mlp-layers_{layer_str}-lr_{lr}"
        run_id = train_and_log(hidden_layer_sizes=layers, learning_rate_init=lr, batch_size=256, max_iter=15, run_name=run_name)
        sweep_run_ids.append(run_id)

print("Sweep run IDs:", sweep_run_ids)

/home/srikar-j-v/courses/DA3408/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Logged run 6d2d7ff2e5ba495f9c63fe5f45810776  |  acc=0.9567  f1=0.9564
🏃 View run mlp-layers_50-lr_0.001 at: http://localhost:5000/#/experiments/1/runs/6d2d7ff2e5ba495f9c63fe5f45810776
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Step 5 — A 5th run using `mlflow.autolog()`
Compare what gets captured automatically vs. what you logged by hand above.

In [ ]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="rf-autolog"):
    model = MLPClassifier(
        hidden_layer_sizes=(100,),
        learning_rate_init=0.001,
        batch_size=256,
        max_iter=15,
        random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    # autolog captures params + many metrics automatically; we can still add a custom one
    mlflow.log_metric("f1_macro", f1_score(y_test, preds, average="macro"))
    autolog_run_id = mlflow.active_run().info.run_id

print("Autolog run:", autolog_run_id)
mlflow.sklearn.autolog(disable=True)  # turn autolog back off for the rest of the notebook

## Step 6 — Find the best run with `mlflow.search_runs()`
No need to open the UI to find the winner — query it directly.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp-comparison"],
    order_by=["metrics.accuracy DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.hidden_layer_sizes", "params.learning_rate_init", "metrics.accuracy", "metrics.f1_macro", "metrics.train_loss"
)]
print(runs_df[display_cols].head(10).to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (accuracy={best_run['metrics.accuracy']:.4f})")

## Step 7 — Open the MLflow UI
1. Go to **http://localhost:5000** in your browser.
2. Open the **iris-classifier** experiment.
3. Select all 5 runs from this notebook (checkboxes on the left) and click **Compare**.
4. Confirm the run the UI ranks highest matches the `best_run` printed above.

---
### ✅ Deliverable checklist
- [ ] Screenshot of the 5-run comparison view in the MLflow UI
- [ ] The `run_id` of the best run (printed above by `mlflow.search_runs()`)
- [ ] One sentence noting what `autolog()` captured that your manual logging didn't (or vice versa)